# GeneralRFM and Fisheye Coordinates

Use fisheye coordinates to place resolution where the physics needs it and inspect the generated support calls.

Navigation: [Index](../index.ipynb) | Previous: [Curvilinear Boundary Conditions](curvilinear_boundary_conditions.ipynb) | Next: [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb)

## Why This Matters
A fisheye radial map clusters grid points where more resolution is useful while still covering a large outer domain.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [Curvilinear Boundary Conditions](curvilinear_boundary_conditions.ipynb). Terms are defined here before they are used.

## Fisheye Map and Metric Residuals
The sample values show how raw radius maps to a stretched radius. The residuals verify that the metric follows from the coordinate map.

In [1]:
import sympy as sp

import nrpy.reference_metric as refmetric
from nrpy.equations.generalrfm import fisheye
from nrpy.infrastructures.BHaH import generalrfm_precompute
from nrpy.infrastructures.BHaH.fisheye import phys_params_to_fisheye

fisheye_map = fisheye.build_fisheye(2)
print("transitions:", fisheye_map.num_transitions)
print("raw coordinates:", fisheye_map.xx)
print("plateau factors:", fisheye_map.a_list)
print("transition centers:", fisheye_map.R_list)
print("transition widths:", fisheye_map.s_list)
print("scale factor:", fisheye_map.c)
print("rbar_unscaled:", fisheye_map.rbar_unscaled)
print("xx_to_CartU:")
for component in fisheye_map.xx_to_CartU:
    print(component)

rfm = refmetric.reference_metric["GeneralRFM_fisheyeN2"]
print("reference metric:", rfm.CoordSystem)
print("provider:", rfm.general_rfm_provider_name)
print("provider metadata:", rfm.general_rfm_provider_meta)
print("reference-metric map:")
for component in rfm.xx_to_Cart:
    print(component)

substitutions = {
    fisheye_map.a_list[0]: sp.Rational(1, 1),
    fisheye_map.a_list[1]: sp.Rational(3, 2),
    fisheye_map.a_list[2]: sp.Rational(2, 1),
    fisheye_map.R_list[0]: sp.Rational(1, 1),
    fisheye_map.R_list[1]: sp.Rational(3, 1),
    fisheye_map.s_list[0]: sp.Rational(2, 5),
    fisheye_map.s_list[1]: sp.Rational(7, 10),
    fisheye_map.c: sp.Rational(1, 1),
}
print("sample radius map:")
for raw_radius in [sp.Rational(1, 4), sp.Rational(1, 1), sp.Rational(2, 1), sp.Rational(4, 1)]:
    value = fisheye_map.rbar_unscaled.subs({fisheye_map.xx[0]: raw_radius, fisheye_map.xx[1]: 0, fisheye_map.xx[2]: 0, **substitutions})
    print(raw_radius, sp.N(value, 12))

for i, j in [(0, 0), (0, 1), (1, 1), (2, 2)]:
    induced_metric_ij = sum(fisheye_map.dCart_dxxUD[k][i] * fisheye_map.dCart_dxxUD[k][j] for k in range(3))
    residual = sp.simplify(fisheye_map.ghatDD[i][j] - induced_metric_ij)
    print(f"residual ({i}, {j}) =", residual)
    assert residual == 0

print("support functions:")
for label, function in [
    ("generalrfm_precompute", generalrfm_precompute.register_CFunction_generalrfm_precompute),
    ("generalrfm_support", generalrfm_precompute.register_CFunctions_generalrfm_support),
    ("fisheye_params_from_physical", phys_params_to_fisheye.register_CFunction_fisheye_params_from_physical_N),
    ("post_params_hook", phys_params_to_fisheye.build_post_params_struct_set_to_default_hook),
]:
    print(label, callable(function))

transitions: 2
raw coordinates: [xx0, xx1, xx2]
plateau factors: [fisheye_a0, fisheye_a1, fisheye_a2]
transition centers: [fisheye_R1, fisheye_R2]
transition widths: [fisheye_s1, fisheye_s2]
scale factor: fisheye_c
rbar_unscaled: fisheye_a2*sqrt(xx0**2 + xx1**2 + xx2**2) + fisheye_s1*(fisheye_a0 - fisheye_a1)*log(cosh((fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1)/cosh((-fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1))/(2*tanh(fisheye_R1/fisheye_s1)) + fisheye_s2*(fisheye_a1 - fisheye_a2)*log(cosh((fisheye_R2 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s2)/cosh((-fisheye_R2 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s2))/(2*tanh(fisheye_R2/fisheye_s2))
xx_to_CartU:
fisheye_c*xx0*(fisheye_a2*sqrt(xx0**2 + xx1**2 + xx2**2) + fisheye_s1*(fisheye_a0 - fisheye_a1)*log(cosh((fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1)/cosh((-fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1))/(2*tanh(fisheye_R1/fisheye_s1)) + fisheye_s2*(fisheye_a1 - fisheye_a2)*log(cosh((

reference metric: GeneralRFM_fisheyeN2
provider: fisheye
provider metadata: {'num_transitions': 2}
reference-metric map:
fisheye_c*xx0*(fisheye_a2*sqrt(xx0**2 + xx1**2 + xx2**2) + fisheye_s1*(fisheye_a0 - fisheye_a1)*log(cosh((fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1)/cosh((-fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1))/(2*tanh(fisheye_R1/fisheye_s1)) + fisheye_s2*(fisheye_a1 - fisheye_a2)*log(cosh((fisheye_R2 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s2)/cosh((-fisheye_R2 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s2))/(2*tanh(fisheye_R2/fisheye_s2)))/sqrt(xx0**2 + xx1**2 + xx2**2)
fisheye_c*xx1*(fisheye_a2*sqrt(xx0**2 + xx1**2 + xx2**2) + fisheye_s1*(fisheye_a0 - fisheye_a1)*log(cosh((fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1)/cosh((-fisheye_R1 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s1))/(2*tanh(fisheye_R1/fisheye_s1)) + fisheye_s2*(fisheye_a1 - fisheye_a2)*log(cosh((fisheye_R2 + sqrt(xx0**2 + xx1**2 + xx2**2))/fisheye_s2)/cosh((-fisheye_R2 

1/4 0.250468824080


1 1.06385200044


2 2.50329428319


4 6.00242433075


residual (0, 0) = 0


residual (0, 1) = 0


residual (1, 1) = 0


residual (2, 2) = 0
support functions:
generalrfm_precompute True
generalrfm_support True
fisheye_params_from_physical True
post_params_hook True


## Interpreting the Output
The mapping output shows how GeneralRFM changes coordinate spacing while preserving the reference-metric machinery. Fisheye maps use this to concentrate resolution where the wave is most interesting.

## Where This Leads
- [Reference-Metric Applications](reference_metric_applications.ipynb)
- [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb)